# Sensitivity analysis: SAD only vs EUC only vs combined SAD+EUC

This notebook compares three spectral-support variants:

1. **SAD only**
2. **Euclidean distance only**
3. **Combined max(SAD, EUC)**, used as the main configuration

The analysis evaluates whether the field-level spectral-support stratification is stable under alternative distance choices. It is **not** designed to select the variant that best predicts semantic inventory errors.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

# =========================================================
# USER CONFIGURATION
# =========================================================

INPUT_DIR = Path("/kaggle/input/datasets/andgu2026/prisma-data2")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Input file containing pixel-level distances to the assigned SIT-class prototype.
# Required columns are listed below in COLS.
INPUT_CSV = INPUT_DIR / "pixel_spectral_distances_full.csv"

# Edit these if your columns have different names.
COLS = {
    "scene": "scene_id",
    "field": "field_id",
    "class": "class_label",
    "sad": "d_sad",
    "euc": "d_euc",
}

VALID_COL = None  # e.g. "is_valid"

MIN_FIELD_PIXELS = 10

PRIORITY_RELIABILITY_Q = 0.10
UNCERTAIN_RELIABILITY_Q = 0.30
PRIORITY_OUTLIER_FRACTION_Q = 0.90

EPS = 1e-12

print("Input:", INPUT_CSV)
print("Output folder:", OUT_DIR)

Input: /kaggle/input/datasets/andgu2026/prisma-data2/pixel_spectral_distances_full.csv
Output folder: /kaggle/working


## Load pixel-level distance table

The notebook expects a pixel-level table where each row is a valid PRISMA--SIT pixel and where SAD and Euclidean distances have already been computed with respect to the prototype of the assigned original SIT class.

Required columns:

```text
scene_id
field_id
class_label
d_sad
d_euc
```

If your columns have different names, edit `COLS` in the configuration cell.

In [9]:
df = pd.read_csv(INPUT_CSV)

print("Columns:")
print(df.columns.tolist())
print()
print("Rows:", len(df))

required = list(COLS.values())
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
        + "\nEdit COLS in the configuration cell to match your file."
    )

if VALID_COL is not None:
    df = df[df[VALID_COL].astype(bool)].copy()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[COLS["scene"], COLS["field"], COLS["class"], COLS["sad"], COLS["euc"]]).copy()

print("Rows after filtering:", len(df))
print("Scenes:", df[COLS["scene"]].nunique())
print("Field-scene units:", df[[COLS["scene"], COLS["field"]]].drop_duplicates().shape[0])
print("Classes:", df[COLS["class"]].nunique())

df.head()

Columns:
['scene_id', 'field_id', 'class_label', 'd_sad', 'd_euc']

Rows: 2498151
Rows after filtering: 2498151
Scenes: 25
Field-scene units: 190559
Classes: 22


,scene_id,field_id,class_label,d_sad,d_euc
0,Scena_1,3045,221,0.087090,0.087063
1,Scena_1,6127,221,0.080761,0.080739
2,Scena_1,3127,221,0.095868,0.095831
3,Scena_1,3148,221,0.085464,0.085437
4,Scena_1,9081,221,0.146918,0.146786


## Core formulas

For each class, SAD and EUC are robustly normalized using the class-specific median and IQR.

\[
z_k(i) = \frac{d_k(i) - m_{k,c}}{\max(\mathrm{IQR}_{k,c}, \epsilon)}.
\]

The support variants are:

\[
s_{\mathrm{SAD}}(i) = \exp[-\max(0, z_{\mathrm{SAD}}(i))]
\]

\[
s_{\mathrm{EUC}}(i) = \exp[-\max(0, z_{\mathrm{EUC}}(i))]
\]

\[
s_{\mathrm{COMB}}(i) = \exp[-\max(0, z_{\mathrm{SAD}}(i), z_{\mathrm{EUC}}(i))].
\]

Field-level scores are computed as the median pixel-level support score within each field-scene unit.

In [11]:
def add_classwise_robust_stats(data, distance_col, class_col, prefix):
    stats = (
        data.groupby(class_col)[distance_col]
        .agg(
            median="median",
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
        )
        .reset_index()
    )
    stats[f"{prefix}_iqr"] = stats["q3"] - stats["q1"]
    stats = stats.rename(columns={
        "median": f"{prefix}_median",
        "q1": f"{prefix}_q1",
        "q3": f"{prefix}_q3",
    })

    out = data.merge(stats, on=class_col, how="left")
    out[f"{prefix}_z"] = (
        (out[distance_col] - out[f"{prefix}_median"])
        / np.maximum(out[f"{prefix}_iqr"], EPS)
    )
    out[f"{prefix}_outlier"] = (
        out[distance_col] > out[f"{prefix}_q3"] + 1.5 * out[f"{prefix}_iqr"]
    )
    return out


def compute_pixel_support_variants(data):
    out = data.copy()
    out = add_classwise_robust_stats(out, COLS["sad"], COLS["class"], "sad")
    out = add_classwise_robust_stats(out, COLS["euc"], COLS["class"], "euc")

    out["z_sad_pos"] = np.maximum(0, out["sad_z"])
    out["z_euc_pos"] = np.maximum(0, out["euc_z"])
    out["z_combined"] = np.maximum.reduce([
        np.zeros(len(out)),
        out["sad_z"].to_numpy(),
        out["euc_z"].to_numpy(),
    ])

    out["support_sad"] = np.exp(-out["z_sad_pos"])
    out["support_euc"] = np.exp(-out["z_euc_pos"])
    out["support_combined"] = np.exp(-out["z_combined"])

    out["outlier_sad"] = out["sad_outlier"]
    out["outlier_euc"] = out["euc_outlier"]
    out["outlier_combined"] = out["sad_outlier"] | out["euc_outlier"]

    return out


def aggregate_to_field(data, support_col, outlier_col, variant_name):
    gcols = [COLS["scene"], COLS["field"]]

    field = (
        data.groupby(gcols)
        .agg(
            class_label=(COLS["class"], lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0]),
            n_pixels=(support_col, "size"),
            support_median=(support_col, "median"),
            support_mean=(support_col, "mean"),
            support_q25=(support_col, lambda x: x.quantile(0.25)),
            outlier_fraction=(outlier_col, "mean"),
        )
        .reset_index()
    )

    field["variant"] = variant_name
    return field


def assign_spectral_strata(field):
    out = field.copy()
    out["flag"] = "eligible"
    out.loc[out["n_pixels"] < MIN_FIELD_PIXELS, "flag"] = "too_small"

    eligible = out["flag"].eq("eligible")

    q10 = out.loc[eligible, "support_median"].quantile(PRIORITY_RELIABILITY_Q)
    q30 = out.loc[eligible, "support_median"].quantile(UNCERTAIN_RELIABILITY_Q)
    q90_out = out.loc[eligible, "outlier_fraction"].quantile(PRIORITY_OUTLIER_FRACTION_Q)

    out.loc[
        eligible & (
            (out["support_median"] <= q10)
            | (out["outlier_fraction"] >= q90_out)
        ),
        "flag"
    ] = "spectrally_atypical"

    eligible_after_atypical = out["flag"].eq("eligible")
    out.loc[
        eligible_after_atypical
        & (out["support_median"] <= q30)
        & (out["outlier_fraction"] < q90_out),
        "flag"
    ] = "intermediate_support"

    out.loc[out["flag"].eq("eligible"), "flag"] = "spectrally_typical"

    thresholds = {
        "q10_support": q10,
        "q30_support": q30,
        "q90_outlier_fraction": q90_out,
    }

    return out, thresholds


def field_key_set(field, flag_value="spectrally_atypical"):
    return set(
        field.loc[
            field["flag"].eq(flag_value),
            [COLS["scene"], COLS["field"]]
        ].apply(tuple, axis=1)
    )


def jaccard(a, b):
    if len(a | b) == 0:
        return np.nan
    return len(a & b) / len(a | b)


def recall_against_main(main_set, variant_set):
    if len(main_set) == 0:
        return np.nan
    return len(main_set & variant_set) / len(main_set)

## Compute variants

In [14]:
pix = compute_pixel_support_variants(df)

variants = {
    "SAD only": ("support_sad", "outlier_sad"),
    "EUC only": ("support_euc", "outlier_euc"),
    "Combined max(SAD,EUC)": ("support_combined", "outlier_combined"),
}

field_tables = {}
thresholds = {}

for name, (support_col, outlier_col) in variants.items():
    field = aggregate_to_field(pix, support_col, outlier_col, name)
    field, thr = assign_spectral_strata(field)
    field_tables[name] = field
    thresholds[name] = thr

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("Field-scene units:", len(field))
    print("Thresholds:", thr)
    print(field["flag"].value_counts(dropna=False))


SAD only
Field-scene units: 190559
Thresholds: {'q10_support': np.float64(0.34422953926376115), 'q30_support': np.float64(0.7282348704874559), 'q90_outlier_fraction': np.float64(0.07142857142857142)}
flag
too_small               149759
spectrally_typical       27863
intermediate_support      7132
spectrally_atypical       5805
Name: count, dtype: int64

EUC only
Field-scene units: 190559
Thresholds: {'q10_support': np.float64(0.3451069818732908), 'q30_support': np.float64(0.7282568400541735), 'q90_outlier_fraction': np.float64(0.07142857142857142)}
flag
too_small               149759
spectrally_typical       27880
intermediate_support      7163
spectrally_atypical       5757
Name: count, dtype: int64

Combined max(SAD,EUC)
Field-scene units: 190559
Thresholds: {'q10_support': np.float64(0.34422953926376115), 'q30_support': np.float64(0.7282267111268885), 'q90_outlier_fraction': np.float64(0.07142857142857142)}
flag
too_small               149759
spectrally_typical       27863
intermed

## Compare variants against the main configuration

The main configuration is the combined maximum of SAD and Euclidean robust deviations.

Metrics:

- **Spearman correlation**: rank correlation of field-level median support scores against the main configuration.
- **Jaccard overlap**: overlap between the spectrally atypical sets.
- **Recall of main atypical set**: fraction of main spectrally atypical field-scene units recovered by the variant.
- **Atypical rate**: fraction of eligible field-scene units flagged as spectrally atypical by the variant.

In [15]:
main_name = "Combined max(SAD,EUC)"
main_field = field_tables[main_name].copy()
main_atypical = field_key_set(main_field, "spectrally_atypical")

rows = []

for name, field in field_tables.items():
    merged = main_field.merge(
        field,
        on=[COLS["scene"], COLS["field"]],
        suffixes=("_main", "_var")
    )

    rho = merged["support_median_main"].corr(
        merged["support_median_var"],
        method="spearman"
    )

    variant_atypical = field_key_set(field, "spectrally_atypical")

    eligible_mask = ~field["flag"].isin(["too_small", "insufficient_class_support"])
    atypical_rate = 100 * (
        field.loc[eligible_mask, "flag"].eq("spectrally_atypical").mean()
    )

    rows.append({
        "Variant": name,
        "Spearman_vs_main": rho,
        "Atypical_Jaccard_vs_main": jaccard(main_atypical, variant_atypical),
        "Recall_of_main_atypical": recall_against_main(main_atypical, variant_atypical),
        "Atypical_rate_pct": atypical_rate,
        "N_atypical": len(variant_atypical),
    })

summary = pd.DataFrame(rows)

summary["order"] = summary["Variant"].map({
    "Combined max(SAD,EUC)": 0,
    "SAD only": 1,
    "EUC only": 2,
})
summary = summary.sort_values("order").drop(columns="order").reset_index(drop=True)

summary

,Variant,Spearman_vs_main,Atypical_Jaccard_vs_main,Recall_of_main_atypical,Atypical_rate_pct,N_atypical
0,"Combined max(SAD,EUC)",1.0,1.000000,1.000000,14.227941,5805
1,SAD only,1.0,1.000000,1.000000,14.227941,5805
2,EUC only,1.0,0.991388,0.991559,14.110294,5757


## Save CSV and LaTeX table

In [17]:
csv_out = OUT_DIR / "sensitivity_sad_euc_combined.csv"
summary.to_csv(csv_out, index=False)

def fmt(x, nd=3):
    if pd.isna(x):
        return "--"
    return f"{x:.{nd}f}"

rows_tex = []
for _, r in summary.iterrows():
    rows_tex.append(
        f"{r['Variant']} & "
        f"{fmt(r['Spearman_vs_main'])} & "
        f"{100 * r['Atypical_Jaccard_vs_main']:.1f} & "
        f"{100 * r['Recall_of_main_atypical']:.1f} & "
        f"{r['Atypical_rate_pct']:.1f} & "
        f"{int(r['N_atypical']):,} \\\\"
    )

latex = "\n".join([
    r"\begin{table}[t]",
    r"\centering",
    r"\scriptsize",
    r"\caption{Sensitivity analysis of spectral-support distance choices. Correlations and overlaps are computed with respect to the main combined SAD--EUC configuration. The analysis evaluates stability of the spectral-atypicality stratification, not semantic error-detection performance.}",
    r"\label{tab:sensitivity_sad_euc}",
    r"\begin{tabular}{lrrrrr}",
    r"\toprule",
    r"Variant & $\rho$ & Jaccard (\%) & Recall (\%) & Atypical (\%) & $n$ atypical \\",
    r"\midrule",
    *rows_tex,
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table}",
])

tex_out = OUT_DIR / "tab_sensitivity_sad_euc.tex"
tex_out.write_text(latex)

print("Saved CSV:", csv_out)
print("Saved LaTeX:", tex_out)
print()
print(latex)

Saved CSV: /kaggle/working/sensitivity_sad_euc_combined.csv
Saved LaTeX: /kaggle/working/tab_sensitivity_sad_euc.tex

\begin{table}[t]
\centering
\scriptsize
\caption{Sensitivity analysis of spectral-support distance choices. Correlations and overlaps are computed with respect to the main combined SAD--EUC configuration. The analysis evaluates stability of the spectral-atypicality stratification, not semantic error-detection performance.}
\label{tab:sensitivity_sad_euc}
\begin{tabular}{lrrrrr}
\toprule
Variant & $\rho$ & Jaccard (\%) & Recall (\%) & Atypical (\%) & $n$ atypical \\
\midrule
Combined max(SAD,EUC) & 1.000 & 100.0 & 100.0 & 14.2 & 5,805 \\
SAD only & 1.000 & 100.0 & 100.0 & 14.2 & 5,805 \\
EUC only & 1.000 & 99.1 & 99.2 & 14.1 & 5,757 \\
\bottomrule
\end{tabular}
\end{table}


## Optional: class-level atypical rates

In [18]:
class_rows = []

for name, field in field_tables.items():
    eligible = field[~field["flag"].isin(["too_small", "insufficient_class_support"])].copy()

    tmp = (
        eligible.groupby("class_label")
        .agg(
            eligible_fields=(COLS["field"], "count"),
            atypical_fields=("flag", lambda x: (x == "spectrally_atypical").sum()),
            median_support=("support_median", "median"),
        )
        .reset_index()
    )
    tmp["atypical_rate_pct"] = 100 * tmp["atypical_fields"] / tmp["eligible_fields"]
    tmp["variant"] = name
    class_rows.append(tmp)

class_summary = pd.concat(class_rows, ignore_index=True)
class_summary = class_summary.sort_values(["class_label", "variant"])

class_csv_out = OUT_DIR / "sensitivity_sad_euc_class_level.csv"
class_summary.to_csv(class_csv_out, index=False)

print("Saved class-level CSV:", class_csv_out)
class_summary.head(20)

Saved class-level CSV: /kaggle/working/sensitivity_sad_euc_class_level.csv


,class_label,eligible_fields,atypical_fields,median_support,atypical_rate_pct,variant
44,221,5415,644,0.990924,11.892890,"Combined max(SAD,EUC)"
22,221,5415,635,0.990924,11.726685,EUC only
0,221,5415,644,0.990927,11.892890,SAD only
45,222,2505,286,1.000000,11.417166,"Combined max(SAD,EUC)"
23,222,2505,286,1.000000,11.417166,EUC only
1,222,2505,286,1.000000,11.417166,SAD only
46,223,11283,1952,1.000000,17.300363,"Combined max(SAD,EUC)"
24,223,11283,1942,1.000000,17.211734,EUC only
2,223,11283,1952,1.000000,17.300363,SAD only
47,231,92,11,0.986896,11.956522,"Combined max(SAD,EUC)"


## Suggested manuscript text

```latex
\subsection{Sensitivity analysis of spectral-support distance choices}
\label{app:sensitivity_sad_euc}

We assessed whether the field-level spectral-support stratification was strongly
dependent on the choice of spectral distance. The main configuration combines
class-wise normalized SAD and Euclidean deviations using the maximum positive
robust deviation. We compared this configuration with two variants based on SAD
only and Euclidean distance only. For each variant, pixel-level support scores
were recomputed, aggregated at the field-scene level using the median, and
stratified using the same global quantile thresholds as the main audit.

This analysis was designed to evaluate the stability of the spectral-atypicality
stratification under reasonable distance choices, not to select the variant that
best predicts semantic inventory errors.
Table~\ref{tab:sensitivity_sad_euc} reports the resulting score correlations,
overlap of spectrally atypical field-scene units, and atypical rates.
```